# Controlling MODFLOW 6 with the API — E. Streamflow augmentation

This is part E of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle and [**C. Adjusting recharge with a callback**](modflow-api-C.ipynb) for the callback mechanism this example builds on.

Use the API to implement a real-time operating rule: keep simulated flow at the southern stream gauge above a minimum threshold by pumping a prediction well only when needed. Simulate a baseline with the well turned off, re-run with a callback that switches the well on whenever streamflow drops too low, and compare the two.

Run the imports and library-location cells below first.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from modflowapi import Callbacks, run_simulation
from notebook_helpers import find_mf6_libraries

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Augment streamflow with the prediction well

A more realistic application: use the API to implement a real-time operating rule. The goal is to keep simulated flow at the southern stream gauge above a minimum threshold by pumping a prediction well only when needed. First simulate a baseline scenario with the well turned off, then re-run with a callback that switches the well on whenever streamflow drops below the minimum, and finally compare the two.

### Simulate the baseline scenario

Run the advanced model with the prediction well turned off to establish the un-augmented streamflow at the southern gauge. The head map shows the model boundaries (stream, pumping wells, and prediction well), and the streamflow is collected into a dataframe for later comparison.

> **A note on units.** This model's time unit is **days**, so MODFLOW reports stream and well flows in cubic feet per *day*. The cells below convert those to the conventional **cubic feet per second (cfs)** by dividing by `86400` (the number of seconds in a day), and multiply back by `86400` when handing a cfs rate to MODFLOW. The division is by `-86400` rather than `86400` simply to flip the sign into a positive-discharge convention. Separately, the value `1e30` that shows up in head arrays is MODFLOW's placeholder for dry or inactive cells; the map cell replaces it with a representative head so the contours plot cleanly.

In [ ]:
# run the base model
start_date = pd.to_datetime("1962-01-01 00:00:00")

# this model's time unit is days, so MODFLOW reports flows in ft^3/day; use this
# to convert to/from the conventional ft^3/second (cfs)
SECONDS_PER_DAY = 86400.0

Before processing any results, define three small helper functions used throughout the rest of the notebook:

- `sfr_gage_flows(gwf)` converts a run's SFR gauge observations to cfs (the units note above),
- `prediction_well_rate(gwf)` reads the prediction well's pumping rate from the budget (this one only matters later, in the streamflow-augmentation example), and
- `add_start_row(df)` prepends a start-of-simulation row so a plotted series starts at 1962.

Defining them once here keeps each result-processing cell short and identical from one scenario to the next; you do not need to study them now, just know they exist.

In [ ]:
def sfr_gage_flows(gwf):
    """SFR gauge observations as a dataframe, converted from ft^3/day to cfs
    (and sign-flipped to a positive-discharge convention)."""
    df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
    df["RIV-FLOW"] /= -SECONDS_PER_DAY
    df["RIV-SWGW"] /= -SECONDS_PER_DAY
    df["TOTAL"] = df["RIV-SWGW"]
    Q0 = df["TOTAL"].iloc[0]
    df["PCT_DIFF"] = -100.0 * (df["TOTAL"] - Q0) / Q0
    return df


def prediction_well_rate(gwf):
    """Prediction-well (augmentation) rate per output time in cfs, read from
    the cell-by-cell budget; NaN when the well is off."""
    cobj = gwf.output.budget()
    rates = []
    for totim in cobj.get_times():
        rec = cobj.get_data(totim=totim, paknam2="prediction")[0]
        rates.append(0.0 if len(rec) == 0 else float(rec["q"][0]))
    rates = np.array(rates) / -SECONDS_PER_DAY
    rates[rates == 0.0] = np.nan
    return rates


def add_start_row(df):
    """Prepend a NaN row at the simulation start date so a plotted series begins
    at the model start (1962) rather than at the first sampled output."""
    start_row = pd.DataFrame({start_date: {c: np.nan for c in df.columns}}).T
    start_row["totim"] = 0.0
    return pd.concat([start_row, df])

In [ ]:
ws = pl.Path(f"../data/synthetic-valley/synthetic-valley-advanced-{sample_frequency}")
new_ws = pl.Path(f"models/synthetic-valley-advanced-{sample_frequency}_api_E")


sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=ws, exe_name=mf6_exe, write_headers=False
)
gwf = sim.get_model("sv")
pak = gwf.get_package("prediction")
new_spd = {}
for kper in range(sim.tdis.nper.array):
    spd = pak.stress_period_data.get_data(kper)
    if spd is None:
        continue
    spd["q"] = 0.0
    new_spd[kper] = spd
pak.stress_period_data.set_data(new_spd)

sim.set_sim_path(new_ws)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

In [ ]:
aspect_ratio = 25.0 / 40.0
height = 6.6
width = aspect_ratio * height
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(width, height),
        sharex=True,
        constrained_layout=True,
    )
    mv = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax, extent=gwf.modelgrid.extent)
    hds = gwf.output.head().get_data()
    # fill dry/inactive cells (flagged 1e30) with a representative head so the
    # contours plot cleanly
    hds[hds == 1e30] = hds[hds != 1e30].mean()
    mv.plot_array(hds)
    CS = mv.contour_array(hds, levels=np.arange(1, 14, 1), colors="black")
    ax.clabel(CS, CS.levels, fmt="%.0f", fontsize=10)
    mv.plot_bc("SFR", color="green")
    mv.plot_bc(
        "WELL",
        package=gwf.pwell,
        plotAll=True,
        color="red",
        kper=12,
    )
    mv.plot_bc("WELL", package=gwf.prediction, plotAll=True, color="orange", kper=12)
    mv.plot_grid(lw=0.5, color="black")

    ax.set_xticklabels([])
    ax.set_yticklabels([])

    fig.savefig(fig_path / "sv-head.png", dpi=300, transparent=False)

In [ ]:
base_df = add_start_row(sfr_gage_flows(gwf))
base_df["augmentation rate"] = np.nan  # no augmentation in the baseline
base_df

### Define the streamflow-augmentation rule

Set the minimum acceptable streamflow and write the callback that enforces it. When the downstream SFR flow falls below `min_flow`, the prediction well is turned on at a rate proportional to the shortfall (capped at a maximum), and that water is added as stream inflow at the upstream end of Straight River.

The callback always sets the rate at the start of each stress period using the streamflow from the end of the previous timestep. Setting `update_each_iteration = True` additionally recomputes the rate at the start of every outer iteration (`Callbacks.iteration_start`), so the augmentation tracks the latest simulated streamflow and converges with the rest of the solution instead of lagging it by one timestep.

To keep the per-iteration update stable, the rate is *under-relaxed*: each outer iteration it moves only a fraction `relax` of the way toward the rate the current streamflow calls for. This matters because pumping the prediction well also draws water from the stream, which causes more leakage from the stream and decreases streamflow; under-relaxation lets the rate stabilize and lets the pump switch back off when streamflow recovers.

In [ ]:
min_flow = 15.0  # target minimum streamflow at the gauge (cfs)

# the augmentation period begins after this many stress periods (depends on
# whether the model uses monthly or annual stress periods)
aug_start = 120 if sample_frequency == "monthly" else 10

# rule tuning: required pumping rate = GAIN * (shortfall below min_flow),
# capped at RATE_CAP
GAIN = 2.5  # dimensionless
RATE_CAP = 2e6  # ft^3/day

# when True, the augmentation rate is also recomputed every outer iteration
# (Callbacks.iteration_start) from the latest streamflow; when False, it is set
# once per stress period from the previous timestep's streamflow
update_each_iteration = False

In [ ]:
# run the advanced model in the augmentation workspace through the API with the
# streamflow-augmentation callback defined below (ws0 is populated separately)
sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=new_ws, write_headers=False)

In [ ]:
# the prediction well's augmentation rate (ft^3/d). It persists across outer
# iterations so the per-iteration update can under-relax toward a converged value
# instead of ratcheting up to the cap
aug_rate = 0.0

# under-relaxation factor for the per-iteration update (0 < relax <= 1); smaller
# values damp the change in rate between successive outer iterations
relax = 0.5


def callback_function(sim, callback_step):
    """Augment streamflow with the prediction well, following the rule described
    in the markdown above. `sim` is the running ApiSimulation and `callback_step`
    is the Callbacks stage modflowapi is currently in."""
    global aug_rate

    ml = sim.sv

    def required_rate():
        """Prediction-well rate (ft^3/d) the current streamflow calls for:
        proportional to the shortfall below min_flow (capped), and zero once the
        downstream SFR flow is at or above the target."""
        q = float(
            sim.mf6.get_value(sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1"))[-1]
        )
        q /= SECONDS_PER_DAY
        if q < min_flow:
            return min(GAIN * (min_flow - q) * SECONDS_PER_DAY, RATE_CAP)
        return 0.0

    def apply_rate(rate):
        """Pump the prediction well at `rate` and add the same rate as stream inflow."""
        ml.prediction.stress_period_data["q"] = -rate
        inflow = sim.mf6.get_value_ptr(sim.mf6.get_var_address("INFLOW", "SV", "SFR-1"))
        inflow[0] = rate

    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        # seed the rate once per stress period from the previous timestep's flow
        if sim.kper > aug_start:
            aug_rate = required_rate()
            apply_rate(aug_rate)

    if callback_step == Callbacks.iteration_start:
        # recompute every outer iteration, under-relaxing toward the required rate
        # so it converges with the solution (the rate may rise OR fall) rather than
        # ratcheting up to the cap
        if update_each_iteration and sim.kper > aug_start:
            aug_rate += relax * (required_rate() - aug_rate)
            aug_rate = min(max(aug_rate, 0.0), RATE_CAP)
            apply_rate(aug_rate)

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Compare baseline and augmented streamflow

Collect the augmented streamflow and the well augmentation rate from the model output, then plot the baseline and augmented flows together with the augmentation period and the minimum-flow threshold to see how much the operating rule raises streamflow toward the target.

In [ ]:
gwf = sim.get_model("SV")
df = sfr_gage_flows(gwf)
df["augmentation rate"] = prediction_well_rate(gwf)
df = add_start_row(df)
df

In [ ]:
x0 = df.index[aug_start]
x1 = df.index[-1]
xstart = df.index[1]
xend = df.index[-1]

In [ ]:
base_color, aug_color = "green", "blue"
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2,
        1,
        figsize=(12.3, 4.3),
        sharex=True,
        constrained_layout=True,
    )

    fig.suptitle("Southern Boundary - Gauge 1")

    ax = axs[0]
    y0, y1 = 5, 30
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=base_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Base",
    )
    df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=aug_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.axhline(0, lw=0.5, color="black")
    ax.axhline(min_flow, lw=1.0, ls=":", color="blue", label="Minimum flow")
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("River\nDischarge, cfs")
    leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0 = 0
    if sample_frequency == "annual":
        y1 = 10
    else:
        y1 = 25

    area_df = df["augmentation rate"][xstart:xend].copy()

    area_df.plot(
        ax=ax,
        color="red",
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    for idx in range(1, len(df.index)):
        xa0, xa1 = df.index[idx - 1], df.index[idx]
        ax.fill_between(
            [xa0, xa1],
            0,
            df["augmentation rate"].values[idx],
            color="red",
            lw=1.0,
            edgecolor="black",
            zorder=100,
        )
    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    # leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_southern_boundary.png", dpi=300, transparent=False
    )

**What to look for.** During the shaded augmentation period the blue *Augmented* flow sits above the green *Base* (well-off) flow — the rule is adding water — and the bottom panel shows the prediction-well rate the callback chose: zero until the stream needs help, then rising with the shortfall. Notice, though, that the augmented flow still dips below the dotted `min_flow` line in the driest periods; it does not reach the target, and that is physical rather than a bug. The prediction well is hydraulically connected to the stream, so pumping it pulls water back out of the streambed and removes most of the added water — in test runs the well lifted the gauge only from about 12.3 to 13.3 cfs against the 15 cfs target. The rule supports the flow; at this well location it cannot keep flow at the target on its own.

### Compare the two update strategies

Run the augmentation model twice - once recomputing the rate only at the start of each stress period (`update_each_iteration = False`) and once recomputing it every outer iteration (`update_each_iteration = True`) - and store each result in its own dataframe (`df_period` and `df_iter`). Plotting them together with the baseline shows how recomputing the rate every outer iteration tracks the current streamflow, while updating only once per stress period lags it by a period.

In [ ]:
# run the augmentation model with both update strategies, saving each run to its
# own dataframe (the baseline run is already stored in base_df)
results = {}
for label, flag in (("per stress period", False), ("per iteration", True)):
    update_each_iteration = flag
    run_simulation(lib_name, new_ws, callback_function, verbose=False)

    gwf = sim.get_model("SV")
    rdf = sfr_gage_flows(gwf)
    rdf["augmentation rate"] = prediction_well_rate(gwf)
    results[label] = add_start_row(rdf)

# the two strategies are kept in separate dataframes
df_period = results["per stress period"]
df_iter = results["per iteration"]
df_iter

In [ ]:
idx_ref = df_iter.index
x0, x1 = idx_ref[aug_start], idx_ref[-1]
xstart, xend = idx_ref[1], idx_ref[-1]
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2, 1, figsize=(12.3, 4.3), sharex=True, constrained_layout=True
    )
    fig.suptitle("Southern Boundary - Gauge 1: update-strategy comparison")

    ax = axs[0]
    ax.set_ylim(5, 30)
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="green", lw=1.5, drawstyle=drawstyle, label="Base"
    )
    df_period["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color="orange",
        lw=1.5,
        drawstyle=drawstyle,
        label="Augmented (per stress period)",
    )
    df_iter["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color="blue",
        lw=1.5,
        drawstyle=drawstyle,
        label="Augmented (per iteration)",
    )
    ax.axhline(min_flow, lw=1.0, ls=":", color="black", label="Minimum flow")
    ax.fill_between(
        [x0, x1], 5, 30, color="cyan", alpha=0.25, lw=0, label="Augmentation period"
    )
    ax.set_ylabel("River\nDischarge, cfs")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0 = 0
    if sample_frequency == "annual":
        y1 = 10
    else:
        y1 = 25
    ax.set_ylim(y0, y1)
    df_period["augmentation rate"][xstart:xend].plot(
        ax=ax, color="orange", lw=1.5, drawstyle=drawstyle, label="per stress period"
    )
    df_iter["augmentation rate"][xstart:xend].plot(
        ax=ax, color="blue", lw=1.5, drawstyle=drawstyle, label="per iteration"
    )
    ax.fill_between([x0, x1], y0, y1, color="cyan", alpha=0.25, lw=0)
    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_update_comparison.png", dpi=300, transparent=False
    )

**What to look for.** Recomputing the rate only *per stress period* (orange) sets it from the previous period's flow, so the pump reacts a step late - it can keep pumping into a period that has already recovered, or stay off in one that has just dropped. Recomputing it *per iteration* (blue) under-relaxes the rate toward the current shortfall as the solution converges, so the pump tracks the present streamflow - easing off as flow climbs back toward `min_flow` and ramping up when it falls. The summary table below compares the resulting gauge flow and pumped volume.

In [ ]:
# quick numeric summary over the (shaded) augmentation period


def summarize(df, label):
    rate = df["augmentation rate"].fillna(0.0)  # cfs, 0 when the well is off
    dt_s = df["totim"].diff().fillna(0.0) * SECONDS_PER_DAY  # period length, seconds
    aug = df.iloc[aug_start:]  # rows within the augmentation period
    flow = aug["RIV-FLOW"]
    return pd.Series(
        {
            "min gauge flow (cfs)": flow.min(),
            "mean gauge flow (cfs)": flow.mean(),
            "periods below target": int((flow < min_flow).sum()),
            "max augmentation (cfs)": rate.iloc[aug_start:].max(),
            "total augmented volume (acre-ft)": (rate * dt_s).iloc[aug_start:].sum()
            / 43560.0,
        },
        name=label,
    )


summary = pd.DataFrame(
    [
        summarize(base_df, "base"),
        summarize(df_period, "per stress period"),
        summarize(df_iter, "per iteration"),
    ]
)
summary.round(2)

The table puts numbers on the difference: the per-iteration rule responds to the present streamflow rather than the previous period's. The stream is leaky and some of the groundwater pumped returns to the groundwater from the stream - so neither strategy erases the deficit entirely.

**Recap.** Across these examples you have driven MODFLOW 6 from Python at every level — running a model through `update()`, opening the solver loop by hand, coupling flow and transport, watching convergence live, editing inputs through high-level package objects, and reading and writing internal memory to implement a real-time operating rule. The same lifecycle (`initialize → update`/`solve → finalize`) and the same two access patterns (package objects vs. `get_value` / `get_value_ptr`) underlie every one of them.